# lotte_stance 3-class 분류기 학습 (Phase 5)

**모델:** `klue/roberta-large` fine-tuning (변경 가능)  
**분류:** 3-class CrossEntropyLoss (negative / neutral / positive)  
**입력:** title (seq-A) + description_snippet (seq-B)  
**필터:** is_lotte_related=True 행만 사용  
**max_length:** 256  
**목표:** val macro F1 ≥ 0.75  
**예상 소요:** T4 GPU 기준 약 15~25분

## Kaggle 실행 전 체크리스트

1. **Dataset 업로드**: Kaggle → Datasets → New Dataset 에서 `labeled_titles.csv`, `labeled_players.csv` 업로드
2. **Dataset 연결**: 우측 패널 Add Data → 업로드한 Dataset 추가 (경로: `/kaggle/input/{slug}/`)
3. **GPU 활성화**: Notebook Settings → Accelerator → **GPU T4 x2** 또는 **T4** 선택
4. **인터넷 연결 활성화**: Settings → Internet → On (`klue/roberta-large` HuggingFace 다운로드에 필요)
5. **세션 종료 후**: Output 탭 → `lotte_stance_model.zip` 다운로드

In [ ]:
import torch
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'CUDA: {torch.version.cuda}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('[WARN] GPU 없음 — CPU 학습 시 수 시간 소요')

In [ ]:
# torch, pandas, numpy, scikit-learn은 Kaggle 환경에 사전 설치되어 있음
# 이 패키지들을 재설치하면 RAPIDS(cuml, cudf, dask-cuda)와 버전 충돌 발생
!pip install -q transformers

In [ ]:
# ── 하이퍼파라미터 (필요 시 수정) ─────────────────────────────────────────────
PRETRAINED   = 'klue/roberta-large'   # 대안: 'monologg/koelectra-small-v3-discriminator'
MAX_LENGTH   = 256
BATCH_SIZE   = 8
GRAD_ACCUM   = 2          # effective batch = BATCH_SIZE * GRAD_ACCUM = 16
EPOCHS       = 5
LR           = 3e-5
WARMUP_RATIO = 0.1
SEED         = 42
VAL_SPLIT    = 0.15
SNIPPET_LEN  = 300

# Kaggle 경로
DATA_DIR   = '/kaggle/working/data'
OUTPUT_DIR = '/kaggle/working/lotte_stance_model'

STANCE_LABELS = ['negative', 'neutral', 'positive']
LABEL2ID = {l: i for i, l in enumerate(STANCE_LABELS)}
ID2LABEL = {i: l for i, l in enumerate(STANCE_LABELS)}
VALID_RELATED = {'true', '1', 'yes'}
print('Config 설정 완료')

In [ ]:
# ── Kaggle Dataset에서 CSV 복사 ────────────────────────────────────────────────
# 우측 패널 Add Data에서 연결한 Dataset이 /kaggle/input/{slug}/ 아래에 위치합니다.
import os
import glob
import shutil

os.makedirs(DATA_DIR, exist_ok=True)

for fname in ['labeled_titles.csv', 'labeled_players.csv']:
    candidates = glob.glob(f'/kaggle/input/**/{fname}', recursive=True)
    if candidates:
        src = candidates[0]
        dst = f'{DATA_DIR}/{fname}'
        shutil.copy(src, dst)
        size = os.path.getsize(dst)
        print(f'복사: {src} → {dst}  ({size:,} bytes)')
    else:
        # labeled_players.csv는 선택 파일이므로 WARNING 수준으로 처리
        level = '[ERROR]' if fname == 'labeled_titles.csv' else '[WARN]'
        print(f'{level} {fname} 를 /kaggle/input/ 아래에서 찾을 수 없습니다.')
        if fname == 'labeled_titles.csv':
            print('       Notebook 우측 패널 Add Data → Dataset 연결을 확인하세요.')

In [ ]:
import pandas as pd

totals = {l: 0 for l in STANCE_LABELS}
for fname in ['labeled_titles.csv', 'labeled_players.csv']:
    path = f'{DATA_DIR}/{fname}'
    if not os.path.exists(path):
        print(f'[MISSING] {fname}')
        continue
    df = pd.read_csv(path, encoding='utf-8-sig')
    # lotte_stance 또는 team_stance 컬럼 지원 (하위 호환)
    stance_col = 'team_stance' if 'team_stance' in df.columns and 'lotte_stance' not in df.columns else 'lotte_stance'
    if stance_col not in df.columns:
        print(f'  SKIP: {fname} — lotte_stance/team_stance 컬럼 없음')
        continue
    df_rel = df[df['is_lotte_related'].astype(str).str.lower().isin(VALID_RELATED)].copy()
    df_lbl = df_rel.dropna(subset=[stance_col])
    df_lbl = df_lbl[df_lbl[stance_col].isin(STANCE_LABELS)]
    print(f'{fname}: 총 {len(df)}행 → 학습 가능 {len(df_lbl)}행')
    for l in STANCE_LABELS:
        cnt = (df_lbl[stance_col] == l).sum()
        totals[l] += cnt
        print(f'  {l}: {cnt}')

total = sum(totals.values())
print(f'\n합산 — 학습 예정: {total}행')
for l in STANCE_LABELS:
    print(f'  {l}: {totals[l]} ({totals[l]/max(total,1):.1%})')

In [ ]:
import json
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
)


class StanceDataset(Dataset):
    def __init__(self, titles, snippets, labels, tokenizer):
        self.encodings = tokenizer(
            titles, snippets,
            truncation='only_second', padding='max_length',
            max_length=MAX_LENGTH, return_tensors='pt',
        )
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item


def load_data():
    frames = []
    for fname in ['labeled_titles.csv', 'labeled_players.csv']:
        path = Path(DATA_DIR) / fname
        if not path.exists():
            print(f'  SKIP: {fname} 없음')
            continue
        df = pd.read_csv(path, encoding='utf-8-sig')
        # lotte_stance / team_stance 하위 호환
        stance_col = 'team_stance' if 'team_stance' in df.columns and 'lotte_stance' not in df.columns else 'lotte_stance'
        if stance_col not in df.columns or 'is_lotte_related' not in df.columns:
            print(f'  SKIP: {fname} — 필수 컬럼 없음')
            continue
        before = len(df)
        df = df[df['is_lotte_related'].astype(str).str.lower().isin(VALID_RELATED)].copy()
        df = df.dropna(subset=[stance_col])
        df = df[df[stance_col].isin(STANCE_LABELS)].copy()
        df['_stance'] = df[stance_col]
        print(f'  {fname}: {len(df)}행 로드 (전체 {before}행 중)')
        frames.append(df)

    if not frames:
        raise FileNotFoundError('학습 가능한 lotte_stance 데이터 없음')

    df = pd.concat(frames, ignore_index=True)
    df['title'] = df['title'].fillna('').astype(str).str.strip()
    df['description_snippet'] = (
        df['description_snippet'].fillna('').astype(str)
        .str[:SNIPPET_LEN].str.strip()
    )

    df['_label_id'] = df['_stance'].map(LABEL2ID)
    conflicts = df.groupby('title')['_label_id'].nunique()
    conflicts = conflicts[conflicts > 1].index
    if len(conflicts):
        print(f'  DROP {len(conflicts)}개 충돌 타이틀')
        df = df[~df['title'].isin(conflicts)].reset_index(drop=True)

    dedup_cols = ['title', 'source_name'] if 'source_name' in df.columns else ['title']
    df = df.drop_duplicates(subset=dedup_cols).reset_index(drop=True)

    labels = df['_stance'].map(LABEL2ID).tolist()
    print(f'\n최종 데이터셋: {len(labels)}행')
    for l in STANCE_LABELS:
        print(f'  {l}: {labels.count(LABEL2ID[l])}')
    return df['title'].tolist(), df['description_snippet'].tolist(), labels


def compute_class_weights(labels):
    counts = np.bincount(labels, minlength=len(STANCE_LABELS)).astype(float)
    if np.any(counts == 0):
        missing = [STANCE_LABELS[i] for i, c in enumerate(counts) if c == 0]
        raise ValueError(f'학습 분할에 클래스 없음: {missing}')
    weights = counts.sum() / (len(STANCE_LABELS) * counts)
    weights = np.clip(weights, 0.3, 5.0)
    print(f'Class weights: {dict(zip(STANCE_LABELS, weights.round(3).tolist()))}')
    return torch.tensor(weights, dtype=torch.float)


print('클래스 및 함수 정의 완료')

In [ ]:
def train():
    torch.manual_seed(SEED)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    use_gpu = device.type == 'cuda'
    print(f'Device: {device}\n')

    titles, snippets, labels = load_data()

    tr_t, va_t, tr_s, va_s, tr_l, va_l = train_test_split(
        titles, snippets, labels,
        test_size=VAL_SPLIT, random_state=SEED, stratify=labels,
    )
    print(f'Train: {len(tr_t)}  Val: {len(va_t)}\n')

    print(f'모델 로드: {PRETRAINED} ...')
    tokenizer = AutoTokenizer.from_pretrained(PRETRAINED)
    model = AutoModelForSequenceClassification.from_pretrained(
        PRETRAINED,
        num_labels=len(STANCE_LABELS),
        id2label=ID2LABEL,
        label2id=LABEL2ID,
    ).to(device)

    # Kaggle 환경에서 num_workers > 0 시 DataLoader 데드락 발생 가능
    nw = 0
    train_ds = StanceDataset(tr_t, tr_s, tr_l, tokenizer)
    val_ds   = StanceDataset(va_t, va_s, va_l, tokenizer)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=nw, pin_memory=use_gpu)
    val_loader   = DataLoader(val_ds,   batch_size=16,         shuffle=False, num_workers=nw, pin_memory=use_gpu)

    class_weights = compute_class_weights(tr_l).to(device)
    loss_fn = nn.CrossEntropyLoss(weight=class_weights)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    steps_per_epoch = max(len(train_loader) // GRAD_ACCUM, 1)
    total_steps = steps_per_epoch * EPOCHS
    scheduler = get_linear_schedule_with_warmup(
        optimizer, int(total_steps * WARMUP_RATIO), total_steps,
    )

    best_f1 = 0.0; best_epoch = 0
    Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

    for epoch in range(1, EPOCHS + 1):
        model.train()
        total_loss = 0.0
        optimizer.zero_grad()
        for step, batch in enumerate(train_loader, 1):
            lbl = batch.pop('labels').to(device)
            batch = {k: v.to(device) for k, v in batch.items()}
            loss = loss_fn(model(**batch).logits, lbl) / GRAD_ACCUM
            loss.backward()
            total_loss += loss.item() * GRAD_ACCUM
            if step % GRAD_ACCUM == 0 or step == len(train_loader):
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step(); scheduler.step(); optimizer.zero_grad()

        model.eval()
        all_preds, all_true = [], []
        with torch.no_grad():
            for batch in val_loader:
                lbl = batch.pop('labels')
                batch = {k: v.to(device) for k, v in batch.items()}
                preds = model(**batch).logits.argmax(dim=-1).cpu().tolist()
                all_preds.extend(preds)
                all_true.extend(lbl.tolist())

        macro_f1 = f1_score(all_true, all_preds, average='macro', zero_division=0)
        avg_loss = total_loss / len(train_loader)
        print(f'Epoch {epoch}/{EPOCHS}  loss={avg_loss:.4f}  macro_f1={macro_f1:.4f}')

        if best_epoch == 0 or macro_f1 > best_f1:
            best_f1 = macro_f1; best_epoch = epoch
            model.save_pretrained(OUTPUT_DIR)
            tokenizer.save_pretrained(OUTPUT_DIR)
            with open(f'{OUTPUT_DIR}/stance_config.json', 'w', encoding='utf-8') as f:
                json.dump({'labels': STANCE_LABELS, 'label2id': LABEL2ID, 'id2label': ID2LABEL}, f, indent=2, ensure_ascii=False)
            print('  → Best checkpoint 저장')

    print(f'\n학습 완료 — best macro_f1={best_f1:.4f} (epoch {best_epoch})')

    # Best checkpoint 최종 리포트
    model = AutoModelForSequenceClassification.from_pretrained(OUTPUT_DIR).to(device).eval()
    all_preds, all_true = [], []
    with torch.no_grad():
        for batch in val_loader:
            lbl = batch.pop('labels')
            batch = {k: v.to(device) for k, v in batch.items()}
            all_preds.extend(model(**batch).logits.argmax(dim=-1).cpu().tolist())
            all_true.extend(lbl.tolist())
    print('\nClassification report (best checkpoint):')
    print(classification_report(all_true, all_preds, target_names=STANCE_LABELS, zero_division=0))
    return best_f1


best_f1 = train()

In [ ]:
print('=== 저장된 파일 ===')
for f in sorted(os.listdir(OUTPUT_DIR)):
    size = os.path.getsize(f'{OUTPUT_DIR}/{f}')
    print(f'  {f:<40} {size:>10,} bytes')

cfg = json.load(open(f'{OUTPUT_DIR}/stance_config.json'))
print(f'\nlabel2id: {cfg["label2id"]}')
status = '✓ 달성' if best_f1 >= 0.75 else '✗ 미달 — epoch 증가 또는 lr 조정 고려'
print(f'목표 macro_f1 >= 0.75  {status}')

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

_tok = AutoTokenizer.from_pretrained(OUTPUT_DIR)
_mdl = AutoModelForSequenceClassification.from_pretrained(OUTPUT_DIR).eval()

def predict_stance(title, snippet=''):
    enc = _tok(title, snippet[:SNIPPET_LEN], truncation='only_second',
               padding='max_length', max_length=MAX_LENGTH, return_tensors='pt')
    with torch.no_grad():
        probs = torch.softmax(_mdl(**enc).logits[0], dim=-1).tolist()
    best_idx = max(range(len(probs)), key=lambda i: probs[i])
    return {'label': STANCE_LABELS[best_idx], 'conf': round(probs[best_idx], 4)}

# (title, snippet, expected)
TEST_CASES = [
    ('나균안, 6이닝 2실점 호투로 시즌 5승…롯데 3연패 탈출', '나균안이 두산전 호투로 팀 승리를 이끌었다.', 'positive'),
    ('롯데 불펜 붕괴…9회 4점 내주며 역전패', '롯데 불펜진이 9회에 무너지며 경기를 내줬다.', 'negative'),
    ('롯데 자이언츠, 내일 두산 원정 예고', '롯데는 내일 두산과 잠실 원정을 치른다.', 'neutral'),
    ('전준우 햄스트링 부상, 2주 결장 전망', '전준우가 부상으로 2주 결장할 전망이다.', 'neutral'),
    ('롯데, 외국인 투수 방출…새 용병 물색 중', '롯데가 부진 외국인 투수를 방출했다.', 'negative'),
]

print('=== Smoke Test ===')
passed = 0
for title, snippet, expected in TEST_CASES:
    r = predict_stance(title, snippet)
    ok = r['label'] == expected
    passed += int(ok)
    mark = 'O' if ok else 'X'
    print(f'  {mark} [{r["label"]:>8}] conf={r["conf"]:.3f}  (기대: {expected})')
    print(f'     {title}')
print(f'\n결과: {passed}/{len(TEST_CASES)} 통과')

In [ ]:
# ── 모델 압축 (Kaggle Output 탭에서 다운로드) ─────────────────────────────────
import shutil
import zipfile

ZIP = '/kaggle/working/lotte_stance_model.zip'
shutil.make_archive('/kaggle/working/lotte_stance_model', 'zip', OUTPUT_DIR)
print(f'압축 완료: {ZIP}')
with zipfile.ZipFile(ZIP) as z:
    for name in sorted(z.namelist()):
        print(f'  {name:<45} {z.getinfo(name).file_size:>10,} bytes')
print('\nKaggle 세션 종료 후 우측 Output 탭 → lotte_stance_model.zip 다운로드')

## 다운로드 후 로컬 배치

```
lotte_stance_model.zip 압축 해제
  → training/models/stance_koelectra/
```

필수 파일:
- `config.json`, `model.safetensors` (또는 `pytorch_model.bin`)
- `tokenizer_config.json`, `vocab.txt`
- `stance_config.json` ← 라벨 매핑

환경변수: `STANCE_CLASSIFIER_MODEL_DIR=/app/models/stance_koelectra`